# Day 24 — Lecture 11: Using LassoCV

> Source: Apna College Prime AI/ML • Day 24 Lecture 11 • Video Duration: `06:49`
> Core Objective: `LassoCV` (K-Fold Cross Validation) se automatically best `alpha_` find karna aur `mse` + `r2` evaluate karna.
> Key Parameters: `alphas=a`, `cv=5`, `max_iter=1000`, `random_state=42`
> Dataset Used: `insurance.csv`
> Core Concept: Cross Validation training data ko 5 folds me split karke har `alpha` ka average validation score dekhta hai — isi tarah `RidgeCV` bhi kaam karta hai

## 🛠️ Step 0: Setup & Insurance Feature Pipeline

Chalo `LassoCV` import karte hain aur wahi encoded `insurance.csv` train-test splits ready karte hain.

In [ ]:
# 1. Libraries aur LassoCV import karo
import pandas as pd
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# 2. Dataset load aur prepare karo
insurance_data = pd.read_csv("insurance.csv")
X = insurance_data.drop(columns=["charges"])
y = insurance_data["charges"]
X = pd.get_dummies(X, columns=["region"], drop_first=True, dtype=int)
X["sex"] = X["sex"].map({"female": 1, "male": 0})
X["smoker"] = X["smoker"].map({"yes": 1, "no": 0})
X["age_smoker"] = X["age"] * X["smoker"]
X["bmi_smoker"] = X["bmi"] * X["smoker"]

# 3. Train-test split karo
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 📍 Step 1: K-Fold Cross Validation (`cv=5`) Intuition

Cross Validation pure training data ko `cv=5` barabar **folds** me todta hai:
- Har iteration me 4 folds par train aur 1 alag fold par test hota hai.
- **Note:** Manual loop test set par check karta tha, jabki `LassoCV` training folds ke average par best `alpha_` chunta hai — isliye optimal value alag (`0.001`) aa sakti hai.

In [ ]:
# 1. Training rows aur 5-fold size check karo
print("Total X_train rows:", X_train.shape[0])
print("Approx rows per fold (cv=5):", X_train.shape[0] // 5)

## ⚠️ Step 2: `LassoCV` Fit & `max_iter=1000` Fix

Yaad rakhna:
- **Deprecated Param:** Purane code me `n_alphas` use hota tha, latest sklearn me list dene ke liye `alphas=a` use karte hain.
- **Convergence Warning:** Agar `max_iter` kam ho (jaise `100`), toh Gradient Descent minimum par ruk nahi pata — isliye `max_iter=1000` pass karte hain.

In [ ]:
# 1. Candidate alphas ki list define karo
a = [0.001, 0.1, 1, 2, 5, 10, 20, 30, 40, 50, 100]

# 2. LassoCV model configure aur train karo
lasso_cv_model = LassoCV(
    alphas=a,
    cv=5,
    max_iter=1000,
    random_state=42
)
lasso_cv_model.fit(X_train, y_train)

# 3. Automatically selected best alpha print karo
print("best alpha: ", lasso_cv_model.alpha_)

## 🏆 Step 3: Predictions, MSE & $R^2$ Score (`0.8652`)

Ab `lasso_cv_model` best `alpha_` par trained hai — chalo `X_test` par MSE aur $R^2$ check karte hain. (Bilkul isi code me `RidgeCV` bhi use ho sakta hai!)

In [ ]:
# 1. Test set par predictions lo
y_pred = lasso_cv_model.predict(X_test)

# 2. Mean Squared Error aur R-squared compute karo
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# 3. Final evaluation results print karo
print("mse = ", mse)
print("r2 = ", r2)